# Chronic Patient Monitoring SRA

This notebook consolidates data exploration, preprocessing, anomaly detection, fuzzy clustering, predictive action, SHAP explainability, SARIMA trend checking, and agent-based alerting for the `patient_sra` project.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import shap
import yaml
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == '01_notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / '02_src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from data_loader import load_data
from preprocessing import preprocess_data
from models import run_dbscan, run_fcm, train_predictive_action_layer, train_svm_dynamic, run_sarima
from agents import MessageBus, VitalsAgent, TrendAgent, RiskAgent, AlertAgent
from llm_narrative import generate_clinical_note
from utils import log_to_json

with open(PROJECT_ROOT / 'config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config

## 2. Load Real eICU Data

In [ ]:
df = load_data(config['data']['primary_csv'])
df = df[df['patient_id'].isin(df['patient_id'].unique()[:200])].copy()
df.head()

In [ ]:
df.shape, df.columns.tolist()[:20]

## 3. Preprocess and Engineer Rolling Features

In [ ]:
df = preprocess_data(df)
base_features = ['HR', 'BP', 'SpO2', 'respiration', 'temp', 'creatinine', 'WBC', 'lactate']
df[base_features].describe().T

## 4. DBSCAN and Fuzzy C-Means

In [ ]:
df = run_dbscan(df, base_features, config['model']['dbscan_eps'], config['model']['dbscan_min_samples'])
df = run_fcm(df, base_features)
df[['anomaly_flag', 'prob_stable', 'prob_warning', 'prob_critical']].head()

## 5. Predictive Action Layer and Dynamic Weighting

In [ ]:
drop_cols = {'patient_id', 'hour', 'deterioration_24h', 'deterioration_12h'}
feature_cols = [c for c in df.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])]
X = df[feature_cols].fillna(0)
y_12h = df['deterioration_12h']
y_24h = df['deterioration_24h']

ensemble_model, future_risk_oof = train_predictive_action_layer(X, y_12h)
df['future_risk'] = future_risk_oof
df['dynamic_weight'] = 1 + df['future_risk'] * 5

for _, row in df.head(25).iterrows():
    log_to_json(PROJECT_ROOT / config['logging']['weight_log'], {
        'patient_id': row['patient_id'],
        'hour': int(row['hour']),
        'weight': float(row['dynamic_weight']),
        'justification': f"Future risk is {row['future_risk']:.2f}"
    })

df[['patient_id', 'hour', 'future_risk', 'dynamic_weight']].head()

## 6. Dynamic SVM for 24-Hour Deterioration

In [ ]:
svm_model = train_svm_dynamic(X, y_24h, df['dynamic_weight'])
probs = svm_model.predict_proba(X)[:, 1]
threshold = 0.5
preds = []
y_true = y_24h.astype(int).to_numpy()

for i, p in enumerate(probs, start=1):
    preds.append(int(p >= threshold))
    if i % 100 == 0:
        cm_i = confusion_matrix(y_true[:i], preds)
        if cm_i.size == 4:
            tn_i, fp_i, fn_i, tp_i = cm_i.ravel()
        else:
            tn_i = cm_i[0, 0] if cm_i.shape[0] > 0 else 0
            fp_i = cm_i[0, 1] if cm_i.shape[1] > 1 else 0
        fpr_i = fp_i / (fp_i + tn_i) if (fp_i + tn_i) > 0 else 0
        old_threshold = threshold
        if fpr_i > 0.15:
            threshold = min(0.95, threshold + 0.05)
        elif fpr_i < 0.05:
            threshold = max(0.05, threshold - 0.05)
        log_to_json(PROJECT_ROOT / config['logging']['adaptation_log'], {
            'count': i,
            'fpr': float(fpr_i),
            'old_threshold': float(old_threshold),
            'new_threshold': float(threshold)
        })

preds = np.array(preds)
cm = confusion_matrix(y_true, preds)
acc = accuracy_score(y_true, preds)
auroc = roc_auc_score(y_true, probs) if len(np.unique(y_true)) > 1 else float('nan')
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

print('Accuracy:', round(acc, 4))
print('AUROC:', round(auroc, 4) if not np.isnan(auroc) else auroc)
print('FPR:', round(fpr, 4))
print('Confusion matrix:', cm)

## 7. SHAP, SARIMA, and Agent-Based Alerting

In [ ]:
background = shap.sample(X, min(25, len(X)), random_state=42)
explainer = shap.KernelExplainer(svm_model.predict_proba, background)
shap_values = explainer.shap_values(X.iloc[0:1])
vals = np.abs(shap_values[1][0] if isinstance(shap_values, list) else shap_values[0])
top_features = [X.columns[i] for i in np.argsort(vals)[-3:]]
note = generate_clinical_note(top_features)

bus = MessageBus()
VitalsAgent('VitalsAgent', bus).analyze(df.iloc[-1])
RiskAgent('RiskAgent', bus).analyze(df.iloc[-1]['future_risk'])
AlertAgent('AlertAgent', bus).process_alerts(fpr, threshold)

patient_hr = df[df['patient_id'] == df.iloc[-1]['patient_id']]['HR'].values
if len(patient_hr) >= 24:
    forecast_hr = run_sarima(patient_hr[-24:])
    TrendAgent('TrendAgent', bus).analyze(patient_hr[-1], forecast_hr)

print('Top SHAP features:', top_features)
print('Clinical note:', note)

## 8. Save Dashboard Inputs

In [ ]:
os.makedirs(PROJECT_ROOT / 'logs', exist_ok=True)
out = df[['patient_id', 'hour', 'future_risk']].copy()
out['svm_prob_24h'] = probs
out['svm_pred_24h'] = preds
out['threshold_last'] = threshold
out.to_csv(PROJECT_ROOT / 'logs' / 'predictions.csv', index=False)
out.head()